In [30]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display,update_display
from openai import OpenAI

In [5]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [6]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user","content" : message}]

messages

[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [7]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano",messages=messages)
response.choices[0].message.content

'Hi there! Nice to meet you. I’m here to help with information, explanations, writing, planning, coding, and more. What would you like to do today?\n\nA few quick ideas:\n- Explain a concept in simple terms\n- Draft an email or message\n- Brainstorm ideas or plan a project\n- Summarize an article or help with reading\n- Help with math, coding, or debugging\n- Plan a trip or study schedule\n\nTell me your preferred style (concise or detailed, casual or formal) and what you’d like to work on. What would you like to start with?'

In [8]:
cricbuzz = fetch_website_contents("https://www.cricbuzz.com/")
# print(ed)
print(cricbuzz)

Cricbuzz | Live Cricket Score, Schedule, News, Stats &amp; Videos  | Cricbuzz.com

Menu
Live Scores
Schedule
Archives
News
Series
Teams
Videos
Rankings
More
MATCHES
INDA
vs
AUSA
-
Stumps
AUSU19
vs
INDU19
-
INDU19 won
INDWA
vs
AUSWA
-
AUSWA won
RSA
vs
AUS
-
Preview
ENG
vs
SL
-
Preview
ALL
All
Live Now
Today
INTERNATIONAL
Asian Games 2026
Japan vs Afghanistan
1st Match, Group A
Hong Kong, China vs Malaysia
2nd Match, Group B
SA v AUS 2026
South Africa vs Australia
1st ODI
England vs Sri Lanka, 2026
England vs Sri Lanka
1st ODI
England vs Sri Lanka
2nd ODI
Quadrangular T20I Series in Nigeria, 2026
Ghana vs Sierra Leone
2nd Match
Nigeria vs Sierra Leone
3rd Match
North American Cup, 2026
Cayman Islands vs Bermuda
1st Match (Rescheduled match)
Cayman Islands vs Bermuda
2nd Match
DOMESTIC
Australia A tour of India 2026
India A vs Australia A
1st unofficial Test
Australia U19 tour of India 2026
Australia U19 vs India U19
3rd unofficial ODI
County Championship Division One 2026
Warwickshire vs

In [9]:
system_prompt = """
You are a cricket scorecard summarizer.

When given information about an  cricket match, provide a concise and easy-to-read match summary.

Include:
1. The overall match score, showing both teams' scores, overs, and the match result.
2. The Player of the Match.
3. The main highlights and standout performances, including important batting, bowling, or other key contributions.

Format the response clearly so that the user can quickly understand the final score, who won, who was Player of the Match, and the most important moments of the match.

Do not provide the complete ball-by-ball scorecard. Focus only on the overall score, Player of the Match, and key match highlights.

If the scorecard for the requested match is not yet available because the
match has not started, provide the match's starting details instead,
including the teams, scheduled start time, venue, and competition.
"""

In [10]:
user_prompt_prefix = """
The user wants information about this cricket match:
"""

In [11]:
def messages_for(user_request):
    return [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                user_prompt_prefix
                + user_request)
        }
    ]

In [12]:
messages_for(cricbuzz)

[{'role': 'system',
  'content': "\nYou are a cricket scorecard summarizer.\n\nWhen given information about an  cricket match, provide a concise and easy-to-read match summary.\n\nInclude:\n1. The overall match score, showing both teams' scores, overs, and the match result.\n2. The Player of the Match.\n3. The main highlights and standout performances, including important batting, bowling, or other key contributions.\n\nFormat the response clearly so that the user can quickly understand the final score, who won, who was Player of the Match, and the most important moments of the match.\n\nDo not provide the complete ball-by-ball scorecard. Focus only on the overall score, Player of the Match, and key match highlights.\n\nIf the scorecard for the requested match is not yet available because the\nmatch has not started, provide the match's starting details instead,\nincluding the teams, scheduled start time, venue, and competition.\n"},
 {'role': 'user',
  'content': "\nThe user wants info

In [13]:
def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [14]:
summarize("https://www.cricbuzz.com/")

"Match Summary:\n\nAustralia U19 vs India U19, 3rd unofficial ODI, Australia U19 tour of India 2026\n\n- India U19 won by 6 wickets.\n- Australia U19 scored 254 runs in 49.3 overs.\n- India U19 chased the target, scoring 255 runs for the loss of 4 wickets in 41.2 overs.\n\nPlayer of the Match: Not explicitly mentioned, but likely an Indian batsman given the successful chase with wickets in hand.\n\nKey Highlights:\n- India U19 successfully chased a target of 255 against Australia U19 with 6 wickets remaining.\n- India completed the chase comfortably with over 8 overs to spare, indicating a strong batting performance.\n- Australia U19 posted a solid total of 254, but their bowlers couldn't defend it.\n\nThis was a competitive List A match where India U19 demonstrated effective batting and game management to secure the win in the series decider."

In [16]:
def display_summary(url):
    summary = summarize(url)
    return summary
    # display(Markdown(summary))

In [17]:
display_summary("https://www.cricbuzz.com/")

'Here is the summary of the completed match from the Australia U19 tour of India 2026:\n\n**Match:** Australia U19 vs India U19, 3rd unofficial ODI  \n**Result:** India U19 won by 6 wickets\n\n**Scores:**  \n- Australia U19 scored 254 runs in 49.3 overs  \n- India U19 chased successfully with 255 runs for 4 wickets in 41.2 overs\n\n**Player of the Match:** Not specifically mentioned, but likely to be a key Indian batsman given the successful chase.\n\n**Highlights:**  \n- Australia U19 posted a competitive total of 254 in their innings.  \n- India U19 achieved the target comfortably with 6 wickets in hand and over 8 overs to spare, showing strong batting performance.  \n- The innings was paced well by India U19, indicating good strike rotation and possibly key contributions from multiple batsmen.\n\nIf you want details on standout individual performances or another match, please let me know!'

In [18]:
translate_system_prompt = """
You are a professional translator.

Your task is to translate the provided summarized content accurately into the requested language.

Preserve:
- The original meaning and factual information.
- Names of people, teams, places, and organizations.
- Numbers, scores, dates, and statistics.
- The original Markdown formatting where appropriate.

Do not add new information, explanations, opinions, or assumptions.
Translate only the provided summary.

Return only the translated content.
"""

In [28]:
Translate_user_Prompt = """
Translate this summarized resutl in tamil language
"""

In [25]:
def translate_content(url):
   summary = display_summary(url)
   stream = openai.chat.completions.create(
       model = "gpt-5-nano",
      messages = [
      {"role" : "system","content" : translate_system_prompt},
      {"role" : "user", "content": Translate_user_Prompt + "\n\n" + summary}
   ],
   stream = True
 )
   response = ""
   display_handle = display(Markdown(""), display_id=True)
   for chunk in stream:
      response += chunk.choices[0].delta.content or ''
      update_display(Markdown(response), display_id=display_handle.display_id)



In [31]:
translate_content("https://www.cricbuzz.com/")

போட்டியின் சுருக்கம்:

1. matches: 3ம் அங்கீகரிக்கப்படாத ODI, இந்தியா 2026 ஆஸ்திரேலியா U19 சுற்றுப்பயணம்
   குழுக்கள்: Australia U19 எதிர் India U19
   முடிவு: India U19 6 விக்கெட் வெற்றி பெற்றது
   ஸ்கோர்கள்:
   - Australia U19: 49.3 ஓவர்களில் 254 ரன்கள் அனைத்தும் வெளியேற்றப்பட்டன
   - India U19: 41.2 ஓவர்களில் 255/4

2. Player of the Match: வழங்கப்பட்ட விவரங்களில் குறிப்பல்ல

3. Highlights:
   - India U19 255 இலக்கை 42 ஓவர்களுக்குள் எட்டிச் சிரமமில்லாமல் அடைந்து 6 விக்கெட் வெற்றி பெற்றது.
   - இந்திய பேட்டிங் அணி கடைசிவரை உறுதியான செயல்பாடு காணப்பட்டு, 6 விக்கெட் இழப்புடன் மற்றும் சுமார் 8 ஓవర்கள் மீதி இலக்கை அடைந்தது.
   - Australia U19 சற்று அதிகமான போட்டிப்போட்டியில் 254 ரன்கள் போட்டது, சில நல்ல பேட்டிங் முயற்சிகளை காட்டியது.

மேலும், India A மற்றும் Australia A இடையிலான 1ம் அங்கீகரிக்கப்படாத டெஸ்டின் குறிப்பு இருந்ததாகவும்:
- India A 334 ரன்கள் சேர்த்தது.
- Australia A Day 2 முடிவில் 171 ரன்களால் பின்தங்கி உள்ளது; India Aக்கு 163 ரன்கள் முன்னிலை உள்ளது.

டெஸ்டுக்கான சுருக்கம் அல்லது மேலதிக விவரங்கள் வேண்டுமானால் தெரிவிக்கவும்!